In [1]:
GITHUB_REPO_URL = "https://github.com/24520152-ben/LLM-Generated-Multi-Level-Adversarial-Attacks-on-Vietnamese-Hate-Speech-Detection"
BASELINE_OPTIONS = {
    "model": "xlmr",
    "epochs": 10,
    "device": "cuda:0",
}  # Use cuda:1 for the second GPU.

In [2]:
import json
import os
import subprocess
from pathlib import Path

kaggle_working = Path("/kaggle/working")
project_root = Path.cwd()
if kaggle_working.is_dir():
    project_root = kaggle_working / "ie403"
    if not project_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--", GITHUB_REPO_URL, str(project_root)],
            check=True,
        )
elif not (project_root / "src" / "vihsd_attack").is_dir():
    project_root = project_root.parent

source_dir = project_root / "src"
if not (source_dir / "vihsd_attack").is_dir():
    raise RuntimeError(f"Cannot find src/vihsd_attack in {project_root}.")
env = {
    **os.environ,
    "UV_PROJECT_ENVIRONMENT": str(project_root / ".venv"),
    "PYTHONPATH": str(source_dir),
}
if kaggle_working.is_dir() and not env.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient

        env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

subprocess.run(
    ["uv", "sync", "--locked", "--python", "3.12"],
    cwd=project_root,
    env=env,
    check=True,
)

train_code = """
import json
import sys
from vihsd_attack.classifiers.service import run_baseline
from vihsd_attack.classifiers.config import BaselineConfig

run_baseline(BaselineConfig(**json.loads(sys.argv[1])))
"""
subprocess.run(
    [
        "uv",
        "run",
        "--no-sync",
        "python",
        "-u",
        "-c",
        train_code,
        json.dumps(BASELINE_OPTIONS, default=str),
    ],
    cwd=project_root,
    env=env,
    check=True,
)

Cloning into '/kaggle/working/ie403'...
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: .venv
Resolved 82 packages in 53ms
 Downloaded pygments
 Downloaded nvidia-cufile
 Downloaded underthesea-core
 Downloaded sentencepiece
 Downloaded networkx
 Downloaded pydantic-core
 Downloaded nvidia-cuda-runtime
 Downloaded tokenizers
 Downloaded nvidia-cuda-cccl
 Downloaded hf-xet
 Downloaded sympy
 Downloaded underthesea
 Downloaded cuda-bindings
 Downloaded scikit-learn
 Downloaded pandas
 Downloaded nvidia-cuda-cupti
 Downloaded transformers
 Downloaded numpy
 Downloaded scipy
 Downloaded nvidia-nvjitlink
 Downloaded nvidia-cuda-nvrtc
 Downloaded pyarrow
 Downloaded nvidia-nvshmem-cu13
 Downloaded nvidia-cusparse
 Downloaded nvidia-cusparselt-cu13
 Downloaded nvidia-nccl-cu13
 Downloaded nvidia-cufft
 Downloaded nvidia-cusolver
 Downloaded triton
 Downloaded nvidia-cublas
 Downloaded nvidia-cudnn-cu13
 Downloaded torch
Prepared 79 packages in 48.18s


Training xlmr on cuda:0


xlmr epoch 1/10: 100%|██████████| 1503/1503 [06:06<00:00,  4.10it/s]


Epoch 1: validation macro-F1 = 59.74%


xlmr epoch 2/10: 100%|██████████| 1503/1503 [06:06<00:00,  4.11it/s]


Epoch 2: validation macro-F1 = 64.75%


... (15 outputs left out)

Epoch 10: validation macro-F1 = 65.44%


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2689.23it/s]


test: macro-F1 = 64.90%, accuracy = 85.72%
Saved baseline: /kaggle/working/artifacts/baselines/xlmr/seed-42_20261125T172914Z_c0c97c52


CompletedProcess(args=['uv', 'run', '--no-sync', 'python', '-u', '-c', '\nimport json\nimport sys\nfrom vihsd_attack.classifiers.service import run_baseline\nfrom vihsd_attack.config import BaselineConfig\n\nrun_baseline(BaselineConfig(**json.loads(sys.argv[1])))\n', '{"model": "xlmr", "epochs": 10, "device": "cuda:0"}'], returncode=0)